Intro

Import libraries and load the dataset into a dataframe for analysis.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

df = pd.read_csv("student-mat.csv", sep=";")

# 1 Exploratory Data Analysis
- We calculated summary statistics to quickly understand the grade distribution before creating visual representations. The results are calculated below.

In [ ]:
finalGrades=df["G3"].values

# Calculate summary stats
mean = np.mean(finalGrades)
median = np.median(finalGrades)
standardDeviation = np.std(finalGrades)

print(f"Summary stastics:\n\tMean: {mean}\n\tMedian: {median}\n\tStandard Deviation: {standardDeviation}")

## Visualizations

### Histogram

#### Distribution of final gades

In [ ]:
sns.set_theme()
histogram = sns.displot(data=df, x="G3")

- The grade distribution appears approximately normal, except for a high concentration of failing grades.
---

#### Distribution of failures

In [ ]:
sns.barplot(data=df, x = "failures", y = "G3")

- Most students have zero failures, there appears to be a negative correlation between the number of failures and the number of students receving that number of failures. Or the number of students decreases as the number of failures increases.

#### Mean final grade and intent on higher education

In [ ]:
sns.barplot(data=df, x = "higher", y = "G3")
plt.title("Mean Final Grade by Intent on Higher Education")
plt.xlabel("Intend to pursue highed education")
plt.ylabel("Final grade")

- Students who intend to pursue higher education consistently out perform the students who do not. Of those intending to pursue higher education also have a much lower variability in final grade.

In [ ]:
# Rearrange dataframe for bivariate scatterplot
melted_df = df.melt(
    id_vars=["G3"],
    value_vars=["G1", "G2"],
    var_name="Grade_Type",
    value_name="Grade_Value",
)

# Single scatterplot call using hue
sns.scatterplot(
    data=melted_df, x="Grade_Value", y="G3", hue="Grade_Type", alpha=0.7
)

plt.xlabel("Semesterly Grade")
plt.ylabel("Final Grade")
plt.title("Semesters One & Two vs Final Grade")
plt.show()

There appears to be a positive correlation between semesterly grades with final grades.

- Miguel please explain

In [ ]:
for attribute in df.columns:
    if attribute == "G3" or attribute == "G1" or attribute == "G2":
        continue
    print(df.groupby(attribute)["G3"].mean().to_string() + "\n")

# 2 Data Preprocessing
- We normalized the education of the parents to account for different family structures. For two parent households we average the parents education and in single parent households the value of the custodial parent is assigned.
- The normalization function is displayed below:

In [ ]:
def normalizeParentalEducation(row):
	if row["Pstatus"] == "T":
		return (row["Fedu"] + row["Medu"])/2
	elif row["guardian"] == "mother":
		return row["Medu"]
	elif row["guardian"] == "father":
		return row["Fedu"]
	else:
		return 0
#applu normalization
df["normParentEdu"] = df.apply(normalizeParentalEducation, axis=1)
# discard input categories after normalization
df = df.drop(columns=["Medu","Fedu"]) 

- After normalizing the data set, we performed a Pearson's correlation analysis of each numerical category against the the final grade. We used this to select the categories to train on in subsequent steps.

In [ ]:
correlations = df.corr(method="pearson", numeric_only=True)["G3"]
print(correlations.sort_values(ascending=False))

- Our correlation analysis found that the preceeding semester averages had the strongest correlation with the final grade at 90% and 80% respectively. This confirms the trend displayed by the prior graph.
- The number of past failures displayed a negative correlation of 36%. Our normalized parental education had an 18.82% correlation, the remaining attributes had diminishing correlation with final grades.

- Using only G1 and G2 to predict G3 with linear regression:

## 3 Model Training
- We used a linear regression model based upon semester one and two grades.
- We split our data set 80-20% for the training and testing sets.

In [ ]:
inputTrainSet, inputTestSet, finalGradeTrainSet, finalGradeTestSet = train_test_split(df[["G1", "G2"]], df["G3"], test_size=0.2, random_state=42)
model = LinearRegression().fit(inputTrainSet, finalGradeTrainSet)
finalGradePredictions = model.predict(inputTestSet)

print(type(y_pred[0]))

# for prediction in y_pred:
# 	pass


print("R^2:", r2_score(finalGradeTestSet, finalGradePredictions))
print("MSE:", mean_squared_error(finalGradeTestSet, finalGradePredictions))

## 4. Model Performance Visualization
- Ipsum Lorem